In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os

In [ ]:
cd /content/drive/MyDrive/Datalab/practice

/content/drive/MyDrive/Datalab/practice


In [ ]:
!pip install sdv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 198.5/198.5 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 198.3/198.3 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.5 MB/s eta 0:00:00


# data preprocessing

In [ ]:
import os
import shutil
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim


In [ ]:
df = pd.read_csv("../data/daily-median-income.csv")

In [ ]:
df.head()

,Entity,Code,Year,Median (2021 prices)
0,Albania,ALB,1996,7.419331
1,Albania,ALB,2002,7.117155
2,Albania,ALB,2005,8.300112
3,Albania,ALB,2008,8.939034
4,Albania,ALB,2012,8.768968


In [ ]:
df = df.drop(columns=["Code"])

In [ ]:
df = df.rename(columns={
    "Median (2021 prices)": "value"
})

In [ ]:
df.head()

,Entity,Year,value
0,Albania,1996,7.419331
1,Albania,2002,7.117155
2,Albania,2005,8.300112
3,Albania,2008,8.939034
4,Albania,2012,8.768968


In [ ]:
global_min_year = df["Year"].min()
global_max_year = df["Year"].max()

countries = df["Entity"].unique()
years = range(global_min_year, global_max_year + 1)

full_index = pd.MultiIndex.from_product(
    [countries, years],
    names=["Entity", "Year"]
)

df_full = (
    df
    .set_index(["Entity", "Year"])
    .reindex(full_index)
    .reset_index()
)


In [ ]:
missing_df = df_full[df_full["value"].isna()]

In [ ]:
df_full.head()

,Entity,Year,value
0,Albania,1963,NaN
1,Albania,1964,NaN
2,Albania,1965,NaN
3,Albania,1966,NaN
4,Albania,1967,NaN


In [ ]:
## 수정 필요

In [ ]:
df_init = df_full.copy()
df_numeric = df_init[['value']]

df_numeric = df_numeric.interpolate(axis=0, limit_direction="both") # linear interpolation
df_numeric = df_numeric.fillna(df_numeric.mean()) # extra Nan as mean of every data
df_init['value'] = df_numeric['value']


In [ ]:
df_init

,Entity,Year,value
0,Albania,1963,7.419331
1,Albania,1964,7.419331
2,Albania,1965,7.419331
3,Albania,1966,7.419331
4,Albania,1967,7.419331
...,...,...,...
12154,Zimbabwe,2021,3.047864
12155,Zimbabwe,2022,3.047864
12156,Zimbabwe,2023,3.047864
12157,Zimbabwe,2024,3.047864


# model

In [ ]:
!pip show sdv

Name: sdv
Version: 1.32.0
Summary: Generate synthetic data for single table, multi table and sequential data
Home-page: 
Author: 
Author-email: "DataCebo, Inc." <info@sdv.dev>
License: BSL-1.1
Location: /usr/local/lib/python3.12/dist-packages
Requires: boto3, botocore, cloudpickle, copulas, ctgan, deepecho, graphviz, numpy, pandas, platformdirs, pyyaml, rdt, sdmetrics, tqdm
Required-by: 


In [ ]:
from sdv.single_table import TVAESynthesizer
from sdv.metadata import SingleTableMetadata

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(data=df_init)
metadata.update_column('value', sdtype='numerical')


discrete_cols = df_init.columns
# for col in discrete_cols:
#     metadata.update_column(col, sdtype='categorical')

model = TVAESynthesizer(
    metadata=metadata,
    epochs=20,
    batch_size=32,
    verbose=True
)

model.fit(df_init)




/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:168: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:134: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
Loss: 9.568: 100%|██████████| 20/20 [01:07<00:00,  3.38s/it]


In [ ]:
synthetic_export = model.sample(num_rows=len(df_init))
print(synthetic_export.head())

             Entity  Year      value
0              Iraq  1974  10.157039
1            Belize  1968  16.991430
2           Moldova  1978   6.517588
3  Colombia (urban)  1976   6.896169
4         Palestine  1975   7.188485


In [ ]:
 synthetic_export.sort_values(by=['Entity','Year'])

,Entity,Year,value
7072,Algeria,1963,8.421073
11099,Algeria,1964,8.027701
2193,Algeria,1965,6.785501
6289,Algeria,1965,6.692497
1154,Algeria,1966,8.505096
...,...,...,...
5336,Zimbabwe,2020,4.169817
6013,Zimbabwe,2020,4.286454
11807,Zimbabwe,2021,3.486823
11084,Zimbabwe,2022,4.084243


In [ ]:
from sdv.evaluation.single_table import evaluate_quality

quality_report = evaluate_quality(
    df_init,
    synthetic_export,
    metadata)

print("Overall similarity score:", quality_report)


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 3/3 [00:00<00:00, 108.44it/s]|
Column Shapes Score: 64.88%

(2/2) Evaluating Column Pair Trends: |██████████| 3/3 [00:00<00:00, 61.00it/s]|
Column Pair Trends Score: 49.29%

Overall Score (Average): 57.09%

Overall similarity score: <sdmetrics.reports.single_table.quality_report.QualityReport object at 0x7abad24d7cb0>


# evaluation

In [ ]:
synthetic_export.sort_values(by=['Entity','Year'])

,Entity,Year,value
7072,Algeria,1963,8.421073
11099,Algeria,1964,8.027701
2193,Algeria,1965,6.785501
6289,Algeria,1965,6.692497
1154,Algeria,1966,8.505096
...,...,...,...
5336,Zimbabwe,2020,4.169817
6013,Zimbabwe,2020,4.286454
11807,Zimbabwe,2021,3.486823
11084,Zimbabwe,2022,4.084243


In [ ]:
from sdv.evaluation.single_table import get_column_plot

fig = get_column_plot(
    real_data=df_init,
    synthetic_data=synthetic_export,
    column_name='value',
    metadata=metadata
)

fig.show()

In [ ]:
import numpy as np
import pandas as pd

dt_missing = dt.copy()

# 예: 데이터 5%를 NaN으로 만들기
missing_fraction = 0.05
for col in dt_missing.columns:
    dt_missing.loc[dt_missing.sample(frac=missing_fraction).index, col] = np.nan


from sklearn.metrics import mean_squared_error

col = 'Revenue'
mask = dt_missing[col].isna()
rmse = mean_squared_error(dt[col][mask], synthetic_export[col][mask], squared=False)
print(f'RMSE for {col}:', rmse)


col = 'Entity'
mask = dt_missing[col].isna()
accuracy = (synthetic_export[col][mask] == dt[col][mask]).mean()
print(f'Accuracy for {col}:', accuracy)


NameError: name 'dt' is not defined

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error

# 예시: df = 원본 데이터, synthetic_export = 합성 데이터

# 1. 원본 값이 있는 위치 선택
mask_existing = ~df['value'].isna()  # 원래 값이 있는 곳 True

# 2. 원래 값과 합성 데이터 값 선택
original_values = df.loc[mask_existing, 'value']
synthetic_values = synthetic_export.loc[mask_existing, 'value']

# 3. 비교 테이블 생성
comparison = pd.DataFrame({
    'original': original_values,
    'synthetic': synthetic_values
})
print(comparison.head())  # 확인용

# 4. 통계적 비교 (MSE, RMSE)
mse = mean_squared_error(original_values, synthetic_values)
rmse = mse ** 0.5
print("MSE:", mse)
print("RMSE:", rmse)

# 5. 시각화 비교
plt.figure(figsize=(10,5))
plt.plot(original_values.values, label='Original')
plt.plot(synthetic_values.values, label='Synthetic', alpha=0.7)
plt.legend()
plt.title("Original vs Synthetic Comparison")
plt.show()
